**Lane:** Content Refresh Prioritization

Why this lane?
- The dataset is already oriented toward a content refresh scenario: it contains 90-day search performance signals and a declining-trend label.
- The end user is a content editor or SEO specialist at FlyRank.
- The goal is to answer: "Which page should I refresh first to recover traffic?"
- The rest of the repo — pipeline, capstone template, and Week 2–7 notebooks — all assume a ranking/queue workflow, so this lane fits the full arc.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

1. **Decision:** Which content item should be refreshed first to reverse an observed traffic decline?
2. **Actor / Action:** A content editor or SEO specialist reviews the top-ranked pages and allocates limited refresh resources (rewrite, update metadata, republish).
3. **Cost of a wrong answer:**
   - False positive: a false "high-priority" label wastes editor hours and publishing budget on a page that would not have meaningfully recovered.
   - False negative: skipping a genuinely declining page leaves money on the table — lost traffic, continued rank decay, and a widening gap versus competitors.
4. **Why data / ML helps:** Forty-four performance signals interact in non-linear ways across content types. A hand-written rule is too brittle; a model can learn which combinations actually matter for decline.

**Task type:** Ranking / scoring — the output is a priority score that orders pages from most to least likely to benefit from refresh.  
**Target:** `is_declining_label` = `(trend_direction == "down")` — an observed, data-derived outcome, not a subjective rule.  
**Metric:** precision@K.  

> One-paragraph frame:  
> For a FlyRank content editor, deciding which pages to refresh first, we will build a priority ranking from pseudonymized content performance data (30,000 pages × 44 features), predicting the observed declining-trend label (`is_declining_label`) measured by precision@K. A wrong call costs wasted editorial hours or continued traffic and rank loss. A plain rule is not enough because the signals are many and entangled, and the right priority cutoff shifts across content types. We will claim only decision-support, observed / measured / directional results.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Basic shape
print("Rows (pages):", len(df))
print("Distinct pseudonymized clients:", df["client_id"].nunique())

# Derived label for exploration (prep step will create this column officially)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Declining share (base rate):", round(df["is_declining_label"].mean(), 3))
print()
# Real numbers relevant to the lane
print("Median days since last update:", df["days_since_last_update"].median())
print("Pages with zero clicks in 90d:", round((df["clicks_90d"] == 0).mean(), 3))
print("Pages with avg_position == 0 (no GSC data):", (df["avg_position"] == 0).sum())
print()
print("Decline rate by content type:")
print(df.groupby("content_type")["is_declining_label"].mean().sort_values(ascending=False))

Rows (pages): 30000
Distinct pseudonymized clients: 32
Declining share (base rate): 0.542

Median days since last update: 20.0
Pages with zero clicks in 90d: 0.44
Pages with avg_position == 0 (no GSC data): 1205

Decline rate by content type:
content_type
comparison article    0.572453
keyword article       0.560959
feedly article        0.286737
Name: is_declining_label, dtype: float64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, measured, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**I can claim:**
- I can report decline rates and traffic gaps that are observed / measured in the 90-day performance window.
- I can rank pages by predicted need for refresh and show that the ranking aligns with observed outcomes in a held-out test set.
- I can explain feature importances in plain language to help editors understand the signal behind the score.

**I can never claim:**
- That I "predict Google's algorithm" or prove a causal reason why a page declined.
- That label-derived fields (`trend_direction`, `trend_pct`) are independent evidence — they are the target, so they must stay out of the feature space.
- That I am exposing client-specific names, domains, or private queries — pseudonymous IDs are for grouping and splitting only.